# 지하물류 분석 노트북

저자: HAN JAEHYEOK. 물류 데이터 처리와 지도 표현 코드, 수동 입력값이 포함되어 있습니다. 저장 출력과 실행 횟수는 비어 있으며 전체 실행 검증이 완료된 파이프라인은 아닙니다.

입력 경로는 `../data-source/`입니다. 해당 폴더에 같은 이름의 원자료와 코드에 맞는 라이브러리 환경이 필요하며, 집계 JSON만으로 이 노트북 전체가 실행되지는 않습니다.

지도 2·3에는 수동 상수가 있고 지도3의 물류시설 마커 코드는 주석 처리되어 있습니다. 지도 HTML 4개는 같은 폴더의 `original-map-*.html`에서 확인할 수 있습니다. 자료실 지도의 배달량 원은 반지름 비례, 인터랙티브 지도의 배달량 원은 면적 비례 표현입니다. 지도1의 구별 CJ 기준월은 서로 다르며 경유지 총건수를 택배수요로 단정할 수 없습니다. 물류시설·항만 목록의 기준연도는 2022년입니다. 분석 결과와 연결 제안은 건설 타당성 검증이 아닙니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
import requests
import json
import xmltodict

from folium import plugins
from time import time

In [ ]:
state_geo = 'https://raw.githubusercontent.com/southkorea/seoul-maps/master/kostat/2013/json/seoul_municipalities_geo_simple.json'

df_import = pd.read_csv('../data-source/CJ택배차량 운행량 월별 통계.csv',
                        encoding='cp949')
df_change = df_import[['주소(구)(ADDR)','경유지_총건수(VIA_CNT)']]
df_change.columns = ['name','values']

df_map1 = folium.Map(location=[37.57,127], zoom_start=10.5, tiles='cartodbpositron')
df_map1.choropleth(geo_data=state_geo,
                  name='경유지_총건수(VIA_CNT)',
                  data=df_change,
                  columns=['name', 'values'],
                  key_on='feature.properties.name',
                  fill_color='Blues',
                  fill_opacity=0.7,
                  line_opacity=0.3,
                  color = 'gray',
                  legend_name = 'number of deliveries')

df_map1.save('df_map1.html')


In [ ]:
df_map1

In [ ]:
df_import = pd.read_excel('../data-source/OD별수송실적통계_2020_2020_221218.xlsx')

df_import['경인']=df_import['경기']+df_import['인천']
df_import['충청']=df_import['충북']+df_import['충남']+df_import['대전']+df_import['세종']
df_import['광주전남']=df_import['광주']+df_import['전남']
df_import['대구경북']=df_import['대구']+df_import['경북']
df_import['부울경']=df_import['부산']+df_import['울산']+df_import['경남']

df_merge=df_import.drop(['경기','인천','충북','충남','대전','세종','전남','광주','경북','대구','경남','부산','울산','제주'],axis=1)

df_trans=df_merge.transpose()

df_trans['경인']=df_trans[7]+df_trans[3]
df_trans['충청']=df_trans[9]+df_trans[10]+df_trans[5]+df_trans[16]
df_trans['광주전남']=df_trans[12]+df_trans[4]
df_trans['대구경북']=df_trans[13]+df_trans[2]
df_trans['부울경']=df_trans[1]+df_trans[6]+df_trans[14]

df_final=df_trans.drop([7,3,9,10,5,16,12,4,13,2,1,6,14,15],axis=1)
df_final.to_excel('df_final.xlsx')


In [ ]:
df_map2 = folium.Map(location=[36.5,128], zoom_start=7, tiles='cartodbpositron')

서울 = [37.56,126.98]
경인 = [37.30,126.98]
강원 = [37.76,128.90]
충청 = [36.32,127.43]
전북 = [35.81,127.10]
전남 = [35.15,126.91]
경북 = [35.87,128.59]
부산 = [35.10,129.03]

folium.Circle(서울,radius=38728*2,fill='blue').add_to(df_map2)
folium.Circle(경인,radius=66155*2,fill='blue').add_to(df_map2)
folium.Circle(강원,radius=3167*2,fill='blue').add_to(df_map2)
folium.Circle(충청,radius=18157*2,fill='blue').add_to(df_map2)
folium.Circle(전북,radius=3390*2,fill='blue').add_to(df_map2)
folium.Circle(전남,radius=9140*2,fill='blue').add_to(df_map2)
folium.Circle(경북,radius=14737*2,fill='blue').add_to(df_map2)
folium.Circle(부산,radius=24968*2,fill='blue').add_to(df_map2)

folium.PolyLine([서울,강원],color='red',opacity=0.5,weight=0.28 *2).add_to(df_map2)
folium.PolyLine([서울,전북],color='red',opacity=0.5,weight=0.24 *2).add_to(df_map2)
folium.PolyLine([서울,경인],color='red',opacity=0.5,weight=5.17 *2).add_to(df_map2)
folium.PolyLine([서울,충청],color='red',opacity=0.5,weight=0.86 *2).add_to(df_map2)
folium.PolyLine([서울,전남],color='red',opacity=0.5,weight=0.26 *2).add_to(df_map2)
folium.PolyLine([서울,경북],color='red',opacity=0.5,weight=0.40 *2).add_to(df_map2)
folium.PolyLine([서울,부산],color='red',opacity=0.5,weight=0.53 *2).add_to(df_map2)
folium.PolyLine([강원,전북],color='red',opacity=0.5,weight=0.36 *2).add_to(df_map2)
folium.PolyLine([강원,경인],color='red',opacity=0.5,weight=2.57 *2).add_to(df_map2)
folium.PolyLine([강원,충청],color='red',opacity=0.5,weight=1.92 *2).add_to(df_map2)
folium.PolyLine([강원,전남],color='red',opacity=0.5,weight=0.56 *2).add_to(df_map2)
folium.PolyLine([강원,경북],color='red',opacity=0.5,weight=1.18 *2).add_to(df_map2)
folium.PolyLine([강원,부산],color='red',opacity=0.5,weight=0.91 *2).add_to(df_map2)
folium.PolyLine([전북,경인],color='red',opacity=0.5,weight=1.78 *2).add_to(df_map2)
folium.PolyLine([전북,충청],color='red',opacity=0.5,weight=2.42 *2).add_to(df_map2)
folium.PolyLine([전북,전남],color='red',opacity=0.5,weight=2.00 *2).add_to(df_map2)
folium.PolyLine([전북,경북],color='red',opacity=0.5,weight=0.99 *2).add_to(df_map2)
folium.PolyLine([전북,부산],color='red',opacity=0.5,weight=1.44 *2).add_to(df_map2)
folium.PolyLine([경인,충청],color='red',opacity=0.5,weight=11.98 *2).add_to(df_map2)
folium.PolyLine([경인,전남],color='red',opacity=0.5,weight=2.61 *2).add_to(df_map2)
folium.PolyLine([경인,경북],color='red',opacity=0.5,weight=2.92 *2).add_to(df_map2)
folium.PolyLine([경인,부산],color='red',opacity=0.5,weight=6.54 *2).add_to(df_map2)
folium.PolyLine([충청,전남],color='red',opacity=0.5,weight=2.76 *2).add_to(df_map2)
folium.PolyLine([충청,경북],color='red',opacity=0.5,weight=3.00 *2).add_to(df_map2)
folium.PolyLine([충청,부산],color='red',opacity=0.5,weight=4.22 *2).add_to(df_map2)
folium.PolyLine([전남,경북],color='red',opacity=0.5,weight=1.64 *2).add_to(df_map2)
folium.PolyLine([전남,부산],color='red',opacity=0.5,weight=3.47 *2).add_to(df_map2)
folium.PolyLine([경북,부산],color='red',opacity=0.5,weight=7.10 *2).add_to(df_map2)

df_map2.save('df_map2.html')


In [ ]:
df_map2

In [ ]:
df_import = pd.read_csv('../data-source/2021년 수시교통량.csv',
                        encoding='cp949')

df_sample = df_import[df_import['시간']=='계']
df_sample['중소형화물'] = df_sample['3종'] + df_sample['4종'] + df_sample['5종'] + df_sample['6종'] + df_sample['7종'] 
df_sample['대형화물'] = df_sample['8종'] + df_sample['9종'] + df_sample['10종'] + df_sample['11종'] + df_sample['12종']
df_merge=df_sample.drop(['계','1종','2종','3종','4종','5종','6종','7종','8종','9종','10종','11종','12종'],axis=1)

df_mid = df_merge.sort_values('중소형화물',ascending=False)
df_big = df_merge.sort_values('대형화물',ascending=False)
df_mid.to_excel('df_mid.xlsx')
df_big.to_excel('df_big.xlsx')


In [ ]:
## df_map3 = folium.Map(location=[37.4,127], zoom_start=9.3, tiles='cartodbpositron')
df_map3 = folium.Map(location=[36.5,128], zoom_start=7, tiles='cartodbpositron')

수원신갈 = [37.26,127.10]
하남JCT = [37.53,127.19]
서운JCT = [37.52,126.75]
안성 = [36.99,127.15]
장수 = [37.46,126.75]
중동 = [37.50,126.74]
남동 = [37.43,126.71]
강일 = [37.57,127.16]
토평 = [37.58,127.15]
판교JCT = [37.40,127.09]
송파 = [37.47,127.12]
기흥 = [37.22,127.10]
금호JCT = [35.89,128.52]
도동JCT = [35.91,128.66]
동군포 = [37.32,126.93]
김포 = [37.59,126.77]
북수원 = [37.31,126.99]
남양주 = [37.60,127.15]
마성 = [37.28,127.17]
월곶JCT = [37.38,126.75]
신갈JCT = [37.28,127.10]
상일 = [37.54,127.17]
안성JCT = [37.03,127.13]
송내 = [37.48,126.74]
계양 = [37.54,126.74]
서창JCT = [37.43,126.73]
남양주 = [37.60,127.15]
성남 = [37.42,127.12]
서하남 = [37.51,127.15]
북대구 = [35.90,128.58]
하남 = [37.52,127.21]
부곡 = [37.32,126.96]
노오지JCT = [37.56,126.75]
자유로JCT = [37.61,126.79]
일산 = [37.63,126.80]
구리 = [37.61,127.14]
퇴계원 = [37.64,127.13]
서대구 = [35.87,128.53]
동수원 = [37.30,127.04]
용인 = [37.25,127.20]
김천JCT = [36.15,128.26]
남구미 = [36.06,128.36]
남지 = [35.39,128.48]
창녕 = [35.54,128.47]
청주JCT = [36.56,127.43]
서부산 = [35.15,128.95]
창원JCT = [35.27,128.64]
칠원JCT = [35.28,128.51]
서창JCT = [37.43,126.73]
신탄진 = [36.44,127.41]
구미 = [36.10,128.36]
영산 = [35.45,128.51]
남이JCT = [36.59,127.42]
사상 = [35.15,128.97]
왜관 = [35.97,128.43]
동창원 = [35.27,128.69]
칠서 = [35.32,128.50]
남청주 = [36.53,127.42]
회덕JCT = [36.40,127.41]

folium.PolyLine([수원신갈,신갈JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([하남JCT,상일],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([상일,강일],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([서운JCT,중동],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([안성,안성JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([송내,장수],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([계양,서운JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([남동,서창JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([기흥,수원신갈],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([중동,송내],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([토평,남양주],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([판교JCT,성남],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([송파,서하남],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([서창JCT,월곶JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([북대구,금호JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([하남,하남JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([동군포,부곡],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([김포,노오지JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([도동JCT,북대구],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([서하남,하남JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([자유로JCT,김포],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([일산,자유로JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([노오지JCT,계양],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([남양주,구리],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([구리,퇴계원],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([서대구,금호JCT],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([성남,송파],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([북수원,동수원],color='red',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([마성,용인],color='red',opacity=1/2,weight=10).add_to(df_map3)

folium.PolyLine([구미,김천JCT],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([남구미,구미],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([영산,창녕],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([청주JCT,남이JCT],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([서부산,사상],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([남지,영산],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([왜관,남구미],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([창원JCT,동창원],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([칠원JCT,칠서],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([남청주,청주JCT],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([서창JCT,월곶JCT],color='orange',opacity=1/2,weight=10).add_to(df_map3)
folium.PolyLine([회덕JCT,신탄진],color='orange',opacity=1/2,weight=10).add_to(df_map3)

folium.Marker([35.10,129.04],icon=folium.Icon(color='blue')).add_to(df_map3) ##부산항
folium.Marker([37.45,126.62],icon=folium.Icon(color='blue')).add_to(df_map3) ##인천항

# folium.Marker([37.32,126.93],icon=folium.Icon(color='green')).add_to(df_map3) ##군포IFT
# folium.Marker([37.34,126.70],icon=folium.Icon(color='green')).add_to(df_map3) ##한샘
# folium.Marker([37.32,126.73],icon=folium.Icon(color='green')).add_to(df_map3) ##안산
# folium.Marker([37.14,127.39],icon=folium.Icon(color='green')).add_to(df_map3) ##중부공용
# folium.Marker([37.34,127.10],icon=folium.Icon(color='green')).add_to(df_map3) ##한진
# folium.Marker([37.48,126.65],icon=folium.Icon(color='green')).add_to(df_map3) ##인천트럭
# folium.Marker([37.56,126.60],icon=folium.Icon(color='green')).add_to(df_map3) ##경인아라뱃길인천
# folium.Marker([37.47,127.12],icon=folium.Icon(color='green')).add_to(df_map3) ##서울동남권
# folium.Marker([37.25,127.39],icon=folium.Icon(color='green')).add_to(df_map3) ##이천패션
# folium.Marker([37.04,127.16],icon=folium.Icon(color='green')).add_to(df_map3) ##안성원곡
# folium.Marker([37.59,126.79],icon=folium.Icon(color='green')).add_to(df_map3) ##김포고촌
# folium.Marker([37.03,127.10],icon=folium.Icon(color='green')).add_to(df_map3) ##평택종합
# folium.Marker([37.17,127.13],icon=folium.Icon(color='green')).add_to(df_map3) ##화성동탄
# folium.Marker([37.52,126.77],icon=folium.Icon(color='green')).add_to(df_map3) ##부천오정
# folium.Marker([37.41,127.31],icon=folium.Icon(color='green')).add_to(df_map3) ##광주곤지암
# folium.Marker([37.61,127.30],icon=folium.Icon(color='green')).add_to(df_map3) ##남양주쿠팡
# folium.Marker([37.23,127.29],icon=folium.Icon(color='green')).add_to(df_map3) ##용인대한통운
# folium.Marker([37.15,127.09],icon=folium.Icon(color='green')).add_to(df_map3) ##오산롯데
# folium.Marker([37.74,127.08],icon=folium.Icon(color='green')).add_to(df_map3) ##의정부우편집중
# folium.Marker([37.62,127.14],icon=folium.Icon(color='green')).add_to(df_map3) ##구리쿠팡

## df_map3.save('df_map3-1.html')
df_map3.save('df_map3-2.html')


In [ ]:
df_map3